# B-spline explorer — curvature check and H-conditioning

The pipeline starts from the central sphere congruence, whose radii are
`r = 1/H`. So a reference surface is only usable if its **mean curvature never
vanishes and never changes sign** — an `H` through zero sends the radii through
infinity. This notebook is for checking that before committing to a full run,
and for trying the stage-1b optimisation that pushes `|H|` off zero.

`SURFACES` is every `.json` in `data/bsplines`, so you set `INDEX` to a number
instead of typing a name.


In [14]:
import sys, json
from pathlib import Path

ROOT = Path("..").resolve()
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))
CHAKANA = (ROOT / "Chakana_Geo").resolve()
for sub in ["hanan/src", "kayviz/src"]:
    p = str(CHAKANA / sub)
    if p not in sys.path:
        sys.path.insert(0, p)

import numpy as np
import matplotlib.pyplot as plt
import kayviz as kv

from moebius.pipeline import (
    MoebiusState, load_surface, sample_selection,
    setup_bspline_optimizer, run_bspline_optimizer,
)
from moebius.utils.bsplines import (
    read_bspline_json, save_bspline_to_json,
    sample_bspline_surface, bspline_curvatures,
)

BSPLINE_DIR = ROOT / "data" / "bsplines"

# Every .json in data/bsplines, sorted.  SURFACES[i] is the name without the
# .json, so pick surfaces by number: SURFACES[0], SURFACES[1], ...
SURFACES = sorted(p.stem for p in BSPLINE_DIR.glob("*.json"))

print(f"{len(SURFACES)} surfaces in {BSPLINE_DIR.relative_to(ROOT)}:")
for i, name in enumerate(SURFACES):
    print(f"  {i:3d}  {name}")

kv.init()



168 surfaces in data/bsplines:
    0  CMC
    1  CM_Strip
    2  Catalan
    3  Complex_test_S
    4  Complex_test_S2
    5  Dome
    6  Enner
    7  Ex_1
    8  Example_2
    9  Example_2_inv
   10  Example_3
   11  Example_4
   12  Florian
   13  Lilium
   14  Neg_Surf
   15  Roof
   16  Roof_2
   17  Roof_3
   18  Roof_4
   19  Roof_L
   20  Sample_C
   21  Sph_Inv_2
   22  Sph_inv1
   23  Sph_inv_3
   24  Surfjson
   25  Test_Botanic
   26  Test_Botanic_coarse
   27  Test_Ex1
   28  Test_Mesh
   29  Test_SubD
   30  Test_quad_ex9
   31  Torus
   32  Tunel
   33  Untitled
   34  botanic
   35  catalan_def_1
   36  catalan_def_2
   37  catalan_def_3
   38  ellipsoid
   39  inv_1
   40  last_ex_1
   41  last_ex_1_inv
   42  last_ex_2
   43  last_ex_2_inv
   44  last_ex_3
   45  last_ex_3_inv
   46  last_ex_3_inv_1
   47  last_ex_4
   48  last_ex_4_inv
   49  last_ex_4_inv_1
   50  last_ex_5
   51  last_ex_5_inv
   52  last_ex_5_inv_1
   53  lilium_def
   54  min_c
   55  roof_ex_1
   

---
## 1 — Curvature of one surface

Set `INDEX` to a number from the list above.


In [ ]:
# ── pick a surface ────────────────────────────────────────────────────────────
INDEX    = 27              # index into SURFACES
U_SAMPLE = V_SAMPLE = 20
skipped = [0, 2,6, 10, 11, 12, 13, 14, 17, 18, 19, 20, 21, 23, 24, 25, 26 ]
# ─────────────────────────────────────────────────────────────────────────────
kv.clear()
name = SURFACES[INDEX]
bsp  = read_bspline_json(str(BSPLINE_DIR / f"{name}.json"))
u = np.linspace(0, 1, U_SAMPLE)
v = np.linspace(0, 1, V_SAMPLE)

V, F = sample_bspline_surface(bsp, u, v)
K, H, n = bspline_curvatures(bsp, u, v)

h = H.ravel()
sign_change = h.min() * h.max() < 0
print(f"[{INDEX}] {name}")
print(f"  degree {bsp.order(0)-1}x{bsp.order(1)-1}, control points {bsp.controlpoints.shape[:2]}")
print(f"  H  min {h.min():+.4f}  max {h.max():+.4f}  |H| min {np.abs(h).min():.4e}")
print(f"  K  min {K.min():+.4f}  max {K.max():+.4f}")
print(f"  => {'H CHANGES SIGN - try section 3' if sign_change else 'H is one-signed - usable'}")
if not sign_change and np.abs(h).min() < 1e-2:
    print(f"     but |H| gets as small as {np.abs(h).min():.2e}; r = 1/H will be large there")

kv.register_surface_mesh(name, V, F)
kv.add_scalar_quantity(name, "H", h,          defined_on="vertices")
kv.add_scalar_quantity(name, "K", K.ravel(),  defined_on="vertices")
kv.add_scalar_quantity(name, "|H|", np.abs(h), defined_on="vertices")
kv.add_scalar_quantity(name, "r = 1/H", 1.0 / np.where(np.abs(h) > 1e-9, h, np.nan),
                        defined_on="vertices")
kv.add_vector_quantity(name, "normal", n.reshape(-1, 3), defined_on="vertices")
kv.show()


[27] Test_Ex1
  degree 5x5, control points (40, 40)
  H  min -6.5642  max +14.1513  |H| min 2.2509e-03
  K  min -14.1904  max +12.2987
  => H CHANGES SIGN - try section 3
[kayviz] Viewer already open — http://127.0.0.1:8002/?app=script


---
## 2 — Sweep the whole set

Which surfaces are usable at all. This is the check behind the note in the
README that `data/bsplines/` still needs reducing to surfaces with `H != 0`.

In [ ]:
# ── sweep every surface ───────────────────────────────────────────────────────
SWEEP_SAMPLE = 25            # coarser grid; this is a screen, not a final answer
# ─────────────────────────────────────────────────────────────────────────────

uu = np.linspace(0, 1, SWEEP_SAMPLE)
rows = []                    # (index, H min, H max, |H| min, sign_change or None)
for i, nm in enumerate(SURFACES):
    try:
        b = read_bspline_json(str(BSPLINE_DIR / f"{nm}.json"))
        _, Hs, _ = bspline_curvatures(b, uu, uu)
        hs = Hs.ravel()
        rows.append((i, hs.min(), hs.max(), np.abs(hs).min(),
                     bool(hs.min() * hs.max() < 0)))
    except Exception as e:
        rows.append((i, np.nan, np.nan, np.nan, None))
        print(f"  [{i}] {nm}: {type(e).__name__}: {str(e)[:60]}")

ok  = [r for r in rows if r[4] is False]
bad = [r for r in rows if r[4] is True]
err = [r for r in rows if r[4] is None]
print(f"\n{len(rows)} surfaces: {len(ok)} one-signed H, {len(bad)} sign change, "
      f"{len(err)} failed to read")

print("\nusable:")
for i, lo, hi, amin, _ in ok:
    print(f"  {i:3d}  {SURFACES[i]:22s} |H|min {amin:.3e}   H in [{lo:+.3f}, {hi:+.3f}]")

USABLE = [r[0] for r in ok]          # indices, so they go straight into INDEX
print(f"\nUSABLE = {USABLE}")


In [ ]:
# distribution of the worst |H| per surface — how close to the limit each one is
# rows entries are (index, H min, H max, |H| min, sign_change)
vals = np.array([r[3] for r in rows if r[4] is not None])
flag = np.array([r[4] for r in rows if r[4] is not None])
lo   = np.array([r[1] for r in rows if r[4] is not None])
hi   = np.array([r[2] for r in rows if r[4] is not None])

fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].hist([vals[~flag], vals[flag]], bins=30, stacked=True,
           label=["one-signed H", "H changes sign"])
ax[0].set_xlabel("min |H| over the sampled grid"); ax[0].set_ylabel("surfaces")
ax[0].legend(); ax[0].set_title("how close to H = 0")

ax[1].scatter(lo[~flag], hi[~flag], s=14, label="one-signed H")
ax[1].scatter(lo[flag],  hi[flag],  s=14, label="H changes sign")
ax[1].axhline(0, lw=.6, c="k"); ax[1].axvline(0, lw=.6, c="k")
ax[1].set_xlabel("H min"); ax[1].set_ylabel("H max"); ax[1].legend()
ax[1].set_title("sign change = straddles the origin")
plt.tight_layout(); plt.show()


---
## 3 — Push `|H|` away from zero (stage 1b)

`setup_bspline_optimizer` deforms the surface's **control points** so `|H|`
rises above `H_THRESH`, keeping `r = 1/H` well conditioned. It uses
`MeanCurvatureBspline` with step control and Laplacian smoothing on the control
grid, so the deformation stays gentle.

It obviously helps where `H` merely comes close to zero, but it can also remove
an outright **sign change**, provided the region on the wrong side is small.
`surface_00002` samples `H` over `[-6.84, +6.05]`, and this stage still drove its
12 offending samples across and left `H` entirely negative, moving the surface by
a median of 1.1e-06. What decides the outcome is how much of the surface has to
move, not whether the sign formally changes: a large region of the opposite sign
cannot be nudged out, and the run then either fails to converge or returns
something no longer resembling the input.

So a sign change in section 2 is a reason to try this stage, not to give up.
Judge it by the before/after below — `|H| min`, the sign change, and how far the
surface moved.


In [ ]:
# ── parameters ────────────────────────────────────────────────────────────────
OPT_INDEX  = INDEX           # index into SURFACES; defaults to the one above
H_THRESH   = 0.05            # floor: |H| >= this
H_MAX      = None            # ceiling: |H| <= this, None = off.  Stops the
                             # floor being met by a local dent (a crease);
                             # absolute, so pick ~1x the max |H| printed in
                             # section 1 for this surface.
W_MAX      = 1.0             # weight of the ceiling relative to the floor
W_H        = 1.0
W_STEP     = 0.1             # cap on control-point movement per iteration
OPT_ITER   = 30
# ─────────────────────────────────────────────────────────────────────────────

opt_name = SURFACES[OPT_INDEX]
state = load_surface(opt_name, "bspline_opt",
                     u_sample_num=U_SAMPLE, v_sample_num=V_SAMPLE)

cp_before = state.bspline.controlpoints.copy()
V0, F0 = sample_bspline_surface(state.bspline, state.u_pts, state.v_pts)
_, H0, _ = bspline_curvatures(state.bspline, state.u_pts, state.v_pts)

# H_MAX is an absolute curvature, so it has to be read off this surface.
# The percentiles say where the sharp regions already are: a ceiling near p90
# keeps the bulk of the surface untouched while capping the tail, and one below
# p50 is asking for the whole surface to be flattened.
a0 = np.abs(H0.ravel())
print(f"initial |H|: p50 {np.percentile(a0,50):.3f}  p90 {np.percentile(a0,90):.3f}  "
      f"p99 {np.percentile(a0,99):.3f}  max {a0.max():.3f}")
print(f"  suggested H_MAX ~ {np.percentile(a0,90):.3f} (p90)"
      f"{'   <-- currently off (H_MAX=None)' if H_MAX is None else f'; using {H_MAX}'}")

setup_bspline_optimizer(state, H_thresh=H_THRESH, H_max=H_MAX, w_max=W_MAX,
                        w_H=W_H, w_step=W_STEP)
run_bspline_optimizer(state, max_iter=OPT_ITER)

V1, F1 = sample_bspline_surface(state.bspline, state.u_pts, state.v_pts)
_, H1, _ = bspline_curvatures(state.bspline, state.u_pts, state.v_pts)
h0, h1 = H0.ravel(), H1.ravel()

print(f"\n[{OPT_INDEX}] {opt_name}")
print(f"  |H| min   {np.abs(h0).min():.4e}  ->  {np.abs(h1).min():.4e}   (target {H_THRESH})")
print(f"  H range   [{h0.min():+.4f}, {h0.max():+.4f}]  ->  [{h1.min():+.4f}, {h1.max():+.4f}]")
print(f"  below threshold: {int((np.abs(h0) < H_THRESH).sum())}  ->  {int((np.abs(h1) < H_THRESH).sum())}"
      f"  of {h0.size} samples")
print(f"  sign change: {h0.min()*h0.max() < 0}  ->  {h1.min()*h1.max() < 0}")
print(f"  max |H|   {np.abs(h0).max():.4f}  ->  {np.abs(h1).max():.4f}   (ceiling {H_MAX})")
print(f"  |H| p99.9 {np.percentile(np.abs(h0), 99.9):.4f}  ->  {np.percentile(np.abs(h1), 99.9):.4f}")
print(f"  surface moved: median {np.median(np.linalg.norm(V1-V0, axis=1)):.4e}"
      f"  max {np.linalg.norm(V1-V0, axis=1).max():.4e}")
print(f"  control points moved: max {np.abs(state.bspline.controlpoints - cp_before).max():.4e}")

kv.register_surface_mesh("before", V0, F0)
kv.add_scalar_quantity("before", "H", h0, defined_on="vertices")
kv.register_surface_mesh("after",  V1, F1)
kv.add_scalar_quantity("after", "H", h1, defined_on="vertices")
kv.add_scalar_quantity("after", "moved", np.linalg.norm(V1-V0, axis=1), defined_on="vertices")
kv.show()

In [ ]:
# before / after, as histograms of H
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.hist(h0, bins=60, alpha=.6, label="before")
ax.hist(h1, bins=60, alpha=.6, label="after")
for x in (-H_THRESH, H_THRESH):
    ax.axvline(x, c="k", lw=.8, ls="--")
ax.axvline(0, c="r", lw=1)
ax.set_xlabel("mean curvature H"); ax.set_ylabel("samples")
ax.set_title(f"{opt_name}: dashed = +/- H_THRESH, red = 0")
ax.legend(); plt.tight_layout(); plt.show()

---
## 4 — Save the conditioned surface

Writes the deformed control points back out in the same JSON format, so the
result can be fed to the main pipeline notebook as an ordinary input.

In [ ]:
SAVE_AS = f"{opt_name}_Hfix"      # written to data/bsplines/<SAVE_AS>.json

out = BSPLINE_DIR / f"{SAVE_AS}.json"
save_bspline_to_json(state.bspline, str(out))
print(f"wrote {out}")

# Verify the round trip exactly: normalize=False, because the reader rescales to
# a bounding box of 2 by default and the deformation changed the bounding box, so
# a normalised reload is a *different* surface by design, not a failed write.
raw = read_bspline_json(str(out), normalize=False)
dV = np.abs(np.asarray(raw(state.u_pts, state.v_pts))
            - np.asarray(state.bspline(state.u_pts, state.v_pts))).max()
print(f"  round trip: control points {raw.controlpoints.shape[:2]}, max|dV| {dV:.3e}")
assert dV < 1e-10, "save/read round trip is lossy"

# And what the pipeline itself will see, loading it the normal way.
check = read_bspline_json(str(out))
_, Hc, _ = bspline_curvatures(check, state.u_pts, state.v_pts)
print(f"  as the pipeline loads it: |H| min {np.abs(Hc).min():.4e}, "
      f"sign change {bool(Hc.min() * Hc.max() < 0)}")
print(f"  (|H| is scaled by the normalisation, the sign change is not)")


---
## 5 — Normalise the control points of every file

`read_bspline_json` already normalises on load — it runs Chakana's
`normalize_vertices(cp, factor)`, which centres the control points on the origin
and scales them so the **longest bounding-box dimension equals `factor`**. But
it does that in memory every time, so the files on disk are still at whatever
scale Rhino exported them, and anything reading them with `normalize=False`
gets a different surface.

This writes the normalised control points back, so the stored data matches what
the pipeline actually uses. Two things to know:

- Curvature scales inversely with size, so `H` changes. That is the point of
  doing it once rather than per load, and it is why `H_THRESH` in section 3 is
  comparable across surfaces only after this.
- It is **idempotent**: normalising an already-normalised file leaves it alone,
  because the bounding box is already `factor` and the centre already the
  origin.

`DRY_RUN = True` reports what would change without touching anything. The files
are tracked by git, so a bad run is recoverable with `git checkout`.


In [ ]:
from hanan.geometry.construction import normalize_vertices

# ── parameters ────────────────────────────────────────────────────────────────
FACTOR  = 2.0        # longest bounding-box dimension after normalisation
DRY_RUN = True       # True reports only; set False to rewrite the json files
TOL     = 1e-9       # treat a file as already normalised within this
# ─────────────────────────────────────────────────────────────────────────────

def bbox(cp):
    cp = np.asarray(cp).reshape(-1, 3)
    lo, hi = cp.min(axis=0), cp.max(axis=0)
    return lo, hi, (hi - lo).max()

changed, already, failed = [], [], []
for i, nm in enumerate(SURFACES):
    path = BSPLINE_DIR / f"{nm}.json"
    try:
        raw = read_bspline_json(str(path), normalize=False)
        _, _, span0 = bbox(raw.controlpoints)
        # normalize=True applies normalize_vertices(cp, FACTOR) on load
        nrm = read_bspline_json(str(path), normalize_factor=FACTOR)
        lo, hi, span1 = bbox(nrm.controlpoints)
    except Exception as e:
        failed.append((i, nm, f"{type(e).__name__}: {str(e)[:50]}"))
        continue

    if abs(span0 - FACTOR) < TOL and np.abs(lo + hi).max() < TOL:
        already.append((i, nm))
        continue

    if not DRY_RUN:
        save_bspline_to_json(nrm, str(path))
        # the save must be exact, or every stored surface is quietly altered
        back = read_bspline_json(str(path), normalize=False)
        d = np.abs(np.asarray(back.controlpoints) - np.asarray(nrm.controlpoints)).max()
        assert d < 1e-10, f"{nm}: round trip off by {d:.3e}"
    changed.append((i, nm, span0, span1))

print(f"{'would rewrite' if DRY_RUN else 'rewrote'}: {len(changed)}   "
      f"already normalised: {len(already)}   unreadable: {len(failed)}")
for i, nm, s0, s1 in changed[:12]:
    print(f"  {i:3d}  {nm:22s} longest bbox dim {s0:10.4f} -> {s1:.4f}")
if len(changed) > 12:
    print(f"  ... and {len(changed) - 12} more")
for i, nm, why in failed:
    print(f"  {i:3d}  {nm:22s} SKIPPED - {why}")
if DRY_RUN:
    print("\nDRY_RUN is True - nothing written. Set DRY_RUN = False to apply.")


---
## 6 — Visual triage: keep or skip, one surface at a time

Section 2 screens by a single number. To go through the set by eye instead,
there is a small viewer app with **Keep** and **Skip** buttons:

```bash
python scripts/triage_bsplines.py            # resume where you left off
python scripts/triage_bsplines.py --report   # print the current lists
python scripts/triage_bsplines.py --all      # revisit decided surfaces too
```

Each surface is registered with its `H`, `|H|`, `K` and `r = 1/H` fields, the
buttons are in the viewer's own panel, and `Back` undoes the last decision. The
verdict is written to `data/bsplines_triage.json` on every click, so closing the
tab loses nothing and the next run picks up at the first undecided surface.

It is a **script** rather than a cell because `kv.show()` holds the process for
the whole session, which would block the kernel here. Nothing is deleted:
acting on the skip list is a separate step.
